# SQLite tutorial — first steps for the miRNA knowledgebase

First time using `sqlite3` in Python, first database project. This notebook is a
sandbox: everything runs against throwaway `.db` files in this same `analysis/`
directory (`tutorial_demo.db`, `tutorial_evid.db`), never against a real
`atlas_core.db`. Delete them any time — the last cell does that for you.

Companion documents: `DATABASE_DESIGN_NOTES.txt` §8 (design), `DATABASE_IMPLEMENTATION_PLAN.md`
(what's next). Run cells top to bottom, in order — later cells depend on tables
created earlier.

## 0. Setup — delete any leftover demo files from a previous run

In [1]:
import sqlite3
from pathlib import Path

DEMO_DB = Path("tutorial_demo.db")
EVID_DB = Path("tutorial_evid.db")

for p in (DEMO_DB, EVID_DB):
    p.unlink(missing_ok=True)

print("ready:", DEMO_DB.resolve())

ready: /mnt/raidbio2/extstud/studtemp/mitsopoulos/Text-Mining/analysis/tutorial_demo.db


## 1. The one idea that makes SQLite click

A SQLite database **is a file**. No server, no daemon, no port to start. `atlas_core.db`
will be a file exactly like `mirna_and_disease.assoc` is a file — the difference is the
bytes inside are organised so you can ask questions with SQL instead of scanning line by
line.

`sqlite3` is in the Python standard library — nothing to `pip install`.

## 2. The four things you need

- **Connection** — your open handle to the file: `sqlite3.connect(path)`.
- **`execute(sql, params)`** — run one SQL statement.
- **Cursor** — what you read *results* off of. `execute` returns one.
- **`commit()`** — make writes permanent. Lives on the **connection**, not the cursor.

In [2]:
con = sqlite3.connect(DEMO_DB)          # creates the file if it doesn't exist

con.execute("""
    CREATE TABLE entity (
        id        INTEGER PRIMARY KEY,
        accession TEXT NOT NULL UNIQUE,
        name      TEXT
    ) STRICT
""")

# ? is a placeholder. NEVER build SQL with f-strings / string concatenation.
con.execute("INSERT INTO entity (accession, name) VALUES (?, ?)",
            ("MIMAT0000435", "hsa-miR-21-5p"))

# executemany: one statement, many rows -- the bulk-load workhorse later on
con.executemany("INSERT INTO entity (accession, name) VALUES (?, ?)", [
    ("MONDO_0005070", "neoplasm"),
    ("MONDO_0003220", "breast carcinoma"),
])

con.commit()   # <-- without this, nothing is saved to the file

cur = con.execute("SELECT id, accession, name FROM entity ORDER BY id")
for row in cur:
    print(row)

(1, 'MIMAT0000435', 'hsa-miR-21-5p')
(2, 'MONDO_0005070', 'neoplasm')
(3, 'MONDO_0003220', 'breast carcinoma')


Notice:
- `id INTEGER PRIMARY KEY` auto-assigned 1, 2, 3 — SQLite mints the integer ids for you.
  This is the "integer surrogate id" idea from the design notes.
- Rows come back as **plain tuples** — positional, fragile if you add a column. Fixed in
  section 4 below.
- `STRICT` makes SQLite actually enforce column types. Without it, a string could be
  silently stored in an `INTEGER` column. Always use it.

In [3]:
print("one row  :", con.execute(
    "SELECT name FROM entity WHERE accession=?", ("MONDO_0005070",)
).fetchone())

print("no match :", con.execute(
    "SELECT name FROM entity WHERE accession=?", ("NOPE",)
).fetchone())

one row  : ('neoplasm',)
no match : None


**Gotcha:** `fetchone()` returns `None` when nothing matches — it does not raise. If you
write `name, = con.execute(...).fetchone()` and there's no match, you get a `TypeError`
far away from the actual cause. Check for `None` explicitly when a miss is possible.

## 3. `row_factory` — access columns by name, not position

Plain tuples get unreadable fast once a table has ten columns. `row_factory` fixes that.

In [4]:
con.row_factory = sqlite3.Row

row = con.execute("SELECT * FROM entity WHERE accession=?", ("MIMAT0000435",)).fetchone()
print("by name:", row["accession"], "/", row["name"])
print("keys   :", row.keys())

by name: MIMAT0000435 / hsa-miR-21-5p
keys   : ['id', 'accession', 'name']


## 4. Transactions — all-or-nothing writes

`with con:` wraps a block in a transaction: it commits automatically if the block
succeeds, and rolls back everything in it if an exception is raised partway through.
Below, the second insert violates the `UNIQUE` constraint on `accession` — watch the
first insert (`TEMP_1`) get undone along with it.

In [5]:
try:
    with con:
        con.execute("INSERT INTO entity (accession, name) VALUES (?,?)", ("TEMP_1", "x"))
        con.execute("INSERT INTO entity (accession, name) VALUES (?,?)",
                    ("MIMAT0000435", "dup!"))   # accession already exists -> fails
except sqlite3.IntegrityError as e:
    print("rejected:", e)

n = con.execute("SELECT count(*) FROM entity WHERE accession='TEMP_1'").fetchone()[0]
print("TEMP_1 survived the rollback?", bool(n))

rejected: UNIQUE constraint failed: entity.accession
TEMP_1 survived the rollback? False


## 5. `ATTACH` — querying two files as one

This is the mechanism the whole two-file design (`atlas_core.db` + `atlas_evidence.db`)
runs on. First, build a *second* file standing in for the evidence tier.

In [6]:
evid = sqlite3.connect(EVID_DB)
evid.executescript("""
    CREATE TABLE hit (id INTEGER PRIMARY KEY, entity_id INTEGER, raw_text TEXT) STRICT;
    INSERT INTO hit (entity_id, raw_text) VALUES
        (1, 'miR-21'), (1, 'microRNA-21'), (3, 'breast cancer');
""")
evid.commit()   # executescript returns a CURSOR; commit() is still on the CONNECTION
evid.close()

print("evidence file built:", EVID_DB.resolve())

evidence file built: /mnt/raidbio2/extstud/studtemp/mitsopoulos/Text-Mining/analysis/tutorial_evid.db


In [7]:
con.execute("ATTACH DATABASE ? AS evid", (str(EVID_DB),))

for r in con.execute("""
        SELECT e.name, count(*) AS n
        FROM evid.hit h JOIN main.entity e ON e.id = h.entity_id
        GROUP BY e.name ORDER BY n DESC"""):
    # row_factory (section 3) is still active here, so index by name, not position
    print("joined:", r["name"], "-", r["n"], "mentions")

joined: hsa-miR-21-5p - 2 mentions
joined: breast carcinoma - 1 mentions


`main` is the connection's own file (`tutorial_demo.db`); `evid` is the alias we gave the
attached file. One SQL query, two files, one join — that's the entire mechanism
`atlas_core.db`/`atlas_evidence.db` will use.

## 6. Read-only mode

Two very different reasons to open these files: **building** (read-write) vs.
**using** (read-only — the validator, the UI, a notebook like this one). Read-only
means a stray `INSERT` in an exploratory notebook can never mutate a shipped file.

In [14]:
con.close()   # close the read-write handle first

ro = sqlite3.connect(f"file:{DEMO_DB}?mode=ro", uri=True)
print("reads fine:", ro.execute("SELECT count(*) FROM entity").fetchone()[0], "rows")

try:
    ro.execute("INSERT INTO entity (accession) VALUES ('X')")
except sqlite3.OperationalError as e:
    print("blocked   :", e)

ro.close()

reads fine: 3 rows
blocked   : attempt to write a readonly database


## 7. Why `connect.py` needs a `build_id` check

This is the payoff of the whole tutorial. The two-file design has **no foreign keys
across files** — SQLite cannot express them through `ATTACH`. Integer entity ids in
`atlas_evidence.db` only mean something against the *exact* core build that minted
them.

Below: pretend the core file got rebuilt and one new ontology term was inserted early,
shifting every id after it by one. The evidence file still holds the *old* ids.

In [15]:
con = sqlite3.connect(DEMO_DB)
con.executescript("""
    CREATE TABLE entity_v2 (id INTEGER PRIMARY KEY, accession TEXT, name TEXT) STRICT;
    INSERT INTO entity_v2 (accession, name) VALUES
        ('MONDO_0000001','disease'),          -- NEW term, takes id 1
        ('MIMAT0000435','hsa-miR-21-5p'),     -- was 1, now 2
        ('MONDO_0005070','neoplasm'),         -- was 2, now 3
        ('MONDO_0003220','breast carcinoma'); -- was 3, now 4
""")
con.commit()

con.execute("ATTACH DATABASE ? AS evid", (str(EVID_DB),))

print("Same join query, stale evidence file. No error, no warning:\n")
# fresh connection this cell -- row_factory from section 3 does NOT carry over,
# so this is back to plain tuples
for name, raw_text in con.execute("""SELECT e.name, h.raw_text
                        FROM evid.hit h JOIN main.entity_v2 e ON e.id = h.entity_id"""):
    print("   %-18s <- mention %r" % (name, raw_text))

Same join query, stale evidence file. No error, no warning:

   disease            <- mention 'miR-21'
   disease            <- mention 'microRNA-21'
   neoplasm           <- mention 'breast cancer'


The query **succeeded**. It just told you "miR-21" is evidence for the term **disease**,
and "breast cancer" is evidence for **neoplasm**. Both wrong, and nothing complained.

That's the cost of dropping cross-file foreign keys. The fix is a one-row table copied
into both files at build time:

```sql
CREATE TABLE build_meta (
    build_id       TEXT NOT NULL,   -- e.g. a uuid, generated once per core build
    schema_version INTEGER NOT NULL,
    built_at       TEXT NOT NULL,
    source_run     TEXT             -- which outputs/ directory it came from
) STRICT;
```

`connect.py`, immediately after `ATTACH`, does one check:

```python
core_id = con.execute("SELECT build_id FROM main.build_meta").fetchone()[0]
evid_id = con.execute("SELECT build_id FROM evid.build_meta").fetchone()[0]
if core_id != evid_id:
    raise ValueError(f"mismatched build: core={core_id} evidence={evid_id}")
```

Five lines, and the demo above becomes a loud exception instead of a confidently wrong
answer. Write this *before* the loaders — it's cheap now, expensive to retrofit once
you have a build you don't trust.

## 8. Checklist — what `connect.py` needs to get right

- **read-only vs read-write are separate entry points**, not a flag you might forget
  (`open_core(readonly=True)` style).
- **`PRAGMA foreign_keys = ON`** on every connection — SQLite has foreign keys but they
  are **off by default**, per connection. Forgetting this makes `REFERENCES` decorative.
- **`row_factory = sqlite3.Row`** so callers use column names, not tuple positions.
- **`ATTACH` both tiers, then check `build_id` immediately** (section 7).
- **Bulk-load pragmas** (`PRAGMA journal_mode=WAL`, `synchronous=OFF`) only on the
  build path — they trade durability for speed, fine mid-build, not for a shipped
  read-only file.

### Gotchas that will bite you
- `commit()` is on the **connection**, not the cursor (section 5's `executescript` trap).
- `fetchone()` returns `None` on no match (section 2).
- `?` placeholders always; a one-value tuple needs the trailing comma: `("X",)` not `("X")`.
- `with con:` is a **transaction**, not closing the file — you still call `.close()`.
- `STRICT` on every table, or types silently aren't enforced.

## 9. Where to go next

1. Fix `TEXTMINING_DIR` in `src/textmining/paths.py` (missing `src` in the path — or
   just repoint `DB_DIR = PROJECT_ROOT / "db"`).
2. Write `src/textmining/db/schema/core.sql` with **only** `build_meta` and `entity` —
   nothing else yet.
3. Write `src/textmining/db/connect.py` with `open_core(readonly=)` and the pragmas
   from section 8.
4. Load `entity` from mirBase alone (skip the ontologies for now) — a few thousand
   rows, small enough to eyeball.
5. Open it read-only, run `SELECT count(*) FROM entity`, done — a complete, verifiable
   database at a scale you can check by hand. `association` and its 1.48M rows can
   wait until this plumbing is boring.

## 10. Cleanup — delete the scratch files this notebook made

In [16]:
for p in (DEMO_DB, EVID_DB):
    p.unlink(missing_ok=True)
print("cleaned up")

cleaned up
